# Codification 1 : optimisation, mois d'avril à septembre empilés

**Variante de [`01e_optimisation_tout_regroupe.ipynb`](01e_optimisation_tout_regroupe.ipynb)** (montants et plafond au logarithme signé, cible et codifications regroupées en -2, -1, 0, 2 et plus, même boucle d'optimisation). **Un seul changement : les modèles apprennent aussi sur les mois d'avril à août.** Le problème « retrouver la codification d'un mois à partir de ce que la banque voyait à ce moment » se pose pour chaque mois, et la codification y est connue : chaque client donne jusqu'à cinq exemples (septembre, août, juillet, juin, mai) au lieu d'un, et les cas rares (nouveaux retards, sorties du retard) sont bien plus nombreux. Question : la logique de la banque, apprise sur plusieurs mois, se retrouve-t-elle mieux en septembre ?

**Hypothèse testée, une seule** : la codification 1 est un **code d'attente**. `dpnm` n'est utilisé à aucun moment. Le test reste de côté. Journal : [`tableau_resultats.md`](tableau_resultats.md).

**Choix de construction, à valider avant le lancement** :
1. **Colonnes relatives au mois** : pour un exemple du mois n, `codif_m1` est la codification du mois précédent, `codif_m2` celle d'il y a deux mois… ; `solde_m0` et `paiement_m0` sont le solde et le paiement du mois lui-même, `solde_m1` et `paiement_m1` ceux du mois précédent… ; `LIMIT_BAL` reste le plafond. Ce sont les mêmes valeurs que les colonnes d'origine, rangées par ancienneté : **aucune information nouvelle**. Pour septembre, les variables sont exactement celles du 01e (`codif_m1` = `PAY_2`, `solde_m0` = `BILL_AMT1`…).
2. **Mois avant avril inconnus** : les colonnes trop anciennes sont vides (par exemple, pour mai, il n'existe qu'un mois précédent). Les arbres et CatBoost traitent les vides eux-mêmes ; pour la régression logistique et le MLP, un vide est remplacé par 0 et signalé par un indicateur, à l'intérieur du pipeline. La forme des vides révèle donc le mois aux modèles ; aucune colonne « mois » n'est ajoutée.
3. **Exemples d'apprentissage** : les clients du train de septembre (tous leurs mois) et les clients codifiés 1 en septembre (leurs mois d'avril à août seulement, dont la codification est connue ; jamais leur septembre). Les exemples dont la cible est une codification 1 (34, d'avril à août) sont retirés. **Les clients du test sont entièrement écartés, tous mois confondus.**
4. **Validation** : les mêmes 5 plis que le 01e, par client. Chaque pli est noté **sur le seul mois de septembre** de ses clients, et ces clients sont retirés de l'apprentissage du pli pour tous leurs mois. Les scores se comparent donc directement à ceux du 01e. Le score en train, qui sert au contrôle du surapprentissage, porte sur tous les mois appris.
5. **Départ de la boucle** : les meilleurs réglages du 01e.

## 1. Données

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

df = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")
df = df.drop(columns="dpnm")   # jamais utilisé dans ce notebook
PAY = [f"PAY_{n}" for n in range(1, 7)]
print(f"Clients du niveau 0 : {len(df)} ; colonnes : {df.shape[1]} (dpnm retirée) ; PAY_1 = 1 : {int((df['PAY_1'] == 1).sum())}")

# Codifications 1 d'avril à août : leurs clients ont tous PAY_1 = 1, ils restent donc hors du train et du test
anciens_1 = (df[PAY[1:]] == 1).any(axis=1)
assert (df.loc[anciens_1, "PAY_1"] == 1).all()
print(f"Clients avec une codification 1 d'avril à août : {int(anciens_1.sum())}, tous codifiés 1 aussi en septembre")

## 2. Le découpage de septembre (identique au 01e)

Les clients dont `PAY_1` n'est pas un 1, coupés en 80/20 (train / test, `stratify` sur `PAY_1`, `random_state=42`), exactement comme dans les notebooks 01 à 01e : mêmes clients dans le train, mêmes clients dans le test. Les clients du test ne servent à aucun mois de l'apprentissage. Les variables d'origine retenues sont les mêmes que dans le 01e (codifications d'avril à août, soldes, paiements, plafond ; ni `ID`, ni `dpnm`, ni données personnelles) ; elles sont rangées par mois dans la section 3.

In [ ]:
from sklearn.model_selection import train_test_split

# Variables : colonnes du dataset d'origine (niveau 0), sauf ID (identifiant), PAY_1 (la cible), dpnm et les données personnelles ; aucune colonne ajoutée
PERSONNELLES = ["SEX", "EDUCATION", "MARRIAGE", "AGE"]   # retirées : non pertinentes pour une codification de paiement
NUMERIQUES = ["LIMIT_BAL"] + [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)]
CATEGORIELLES = PAY[1:]
VARIABLES = CATEGORIELLES + NUMERIQUES
assert sorted(VARIABLES + ["ID", "PAY_1"] + PERSONNELLES) == sorted(df.columns)
mask_1 = df["PAY_1"] == 1
connus = df[~mask_1]
a_trancher = df[mask_1]

X_train, X_test, y_train, y_test = train_test_split(
    connus[VARIABLES], connus["PAY_1"], test_size=0.2, stratify=connus["PAY_1"], random_state=42)
CLASSES = sorted(connus["PAY_1"].unique())   # toutes les codifications de PAY_1, sauf 1

repartition = pd.DataFrame({
    "Train": y_train.value_counts(),
    "Test": y_test.value_counts(),
}).reindex(CLASSES).fillna(0).astype(int)
repartition.loc["Total"] = repartition.sum()
print(f"PAY_1 connus : {len(connus)} (train {len(X_train)}, test {len(X_test)}) ; PAY_1 = 1 à trancher : {len(a_trancher)}")
print(f"Variables ({len(VARIABLES)}) : {VARIABLES}")
display(repartition)

**Comment lire** : nombre de clients par codification de `PAY_1`, dans le train et dans le test. Les proportions sont les mêmes dans les deux (`stratify`). Le test n'est plus touché jusqu'au critère A.

## 3. Les mois empilés

Chaque exemple est un client à un mois donné, avec ce que la banque voyait à la fin de ce mois (colonnes relatives, choix 1 de l'introduction) ; la cible est la codification de ce mois, regroupée en -2, -1, 0, 2 et plus. Les codifications des mois précédents sont regroupées de la même façon. Le tableau donne le nombre d'exemples par mois et par codification.

In [ ]:
def regrouper(codifications):
    # 2 et plus -> 2 (retard) ; -2, -1, 0 (et 1) inchangés ; un mois absent (vide) reste vide
    return codifications.mask(codifications >= 2, 2)

def exemples_du_mois(clients, n):
    # Ce que la banque voyait à la fin du mois n : codifications des mois précédents (m1 = mois précédent),
    # soldes et paiements du mois (m0) et des mois précédents ; vide si le mois est antérieur à avril
    d = df.loc[clients]
    bloc = pd.DataFrame(index=d.index)
    for k in range(1, 6):
        bloc[f"codif_m{k}"] = regrouper(d[f"PAY_{n + k}"]) if n + k <= 6 else np.nan
    for k in range(0, 6):
        bloc[f"solde_m{k}"] = d[f"BILL_AMT{n + k}"] if n + k <= 6 else np.nan
        bloc[f"paiement_m{k}"] = d[f"PAY_AMT{n + k}"] if n + k <= 6 else np.nan
    bloc["LIMIT_BAL"] = d["LIMIT_BAL"]
    bloc["cible"] = regrouper(d[f"PAY_{n}"])
    bloc["mois"] = n
    bloc["client"] = d.index
    return bloc

MOIS_NOMS = {1: "septembre", 2: "août", 3: "juillet", 4: "juin", 5: "mai"}
clients_sept = X_train.index                  # clients du train de septembre (PAY_1 connu)
clients_1 = a_trancher.index                  # clients codifiés 1 en septembre : seulement leurs mois d'avril à août
emp = pd.concat([exemples_du_mois(clients_sept, 1)]
                + [exemples_du_mois(clients_sept.union(clients_1), n) for n in range(2, 6)], ignore_index=True)
emp = emp[emp["cible"] != 1].reset_index(drop=True)   # cibles codifiées 1 (avril à août) retirées
assert not emp["client"].isin(X_test.index).any(), "un client du test est dans les exemples"
assert (emp["mois"] == 1).sum() == len(X_train)

CATEGORIELLES = [f"codif_m{k}" for k in range(1, 6)]
NUMERIQUES = [f"solde_m{k}" for k in range(0, 6)] + [f"paiement_m{k}" for k in range(0, 6)] + ["LIMIT_BAL"]
VARIABLES = CATEGORIELLES + NUMERIQUES
X_emp, y_emp = emp[VARIABLES], emp["cible"].astype(int)

# Septembre : mêmes variables que le 01e, dans le même ordre de clients
sept = np.flatnonzero(emp["mois"].to_numpy() == 1)
assert (emp.loc[sept, "codif_m1"].to_numpy() == regrouper(X_train["PAY_2"]).to_numpy()).all()

CLASSES = sorted(y_emp.unique())
tableau = pd.crosstab(emp["mois"].map(MOIS_NOMS).rename("Mois"), y_emp.rename("Codification du mois"), margins=True, margins_name="Total")
print(f"Exemples : {len(emp)} ({emp['client'].nunique()} clients) ; classes : {CLASSES}")
display(tableau.reindex(list(MOIS_NOMS.values()) + ["Total"]))

**Comment lire** : une ligne par mois, une colonne par codification du mois ; septembre ne compte que les clients du train, les autres mois comptent aussi les clients codifiés 1 en septembre. Les mois d'avril à août apportent la plupart des exemples, et en particulier des retards.

## 4. Le score optimisé : la moyenne des quatre familles (choix validé)

- **`"familles équilibrées"`** (retenu) : la moyenne des taux de bonne famille de -2, -1, 0 et 2 et plus. Chaque famille pèse autant : la boucle est poussée à mieux reconnaître les retards, là où tous les modèles butent ;
- **`"bonnes prédictions"`** : la part des clients rangés dans la bonne famille (le score global des notebooks 01 et 01b). Le 0, 56 % des clients, y pèse le plus. Non retenu : les codifications 1 à trancher ne ressemblent pas au train (environ la moitié suivent un retard en août, contre environ 10 % des `PAY_1` connus) ; un modèle réglé sur ce score pencherait vers les codifications saines, majoritaires dans le train. Il reste affiché dans les tableaux de la section 8.

In [ ]:
# ==============================================================================
# Score optimisé (choix validé) : "familles équilibrées" ; autre possibilité : "bonnes prédictions"
# ==============================================================================
SCORE = "familles équilibrées"
SURAPPRENTISSAGE_MAX = 7.0   # écart relatif train - validation, en %
VARIANCE_MAX = 2.0           # écart-type des 5 plis, en points
MAX_TOURS = 10               # tours de la boucle, par modèle

import time
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, make_scorer
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

FAMILLES = [-2, -1, 0, 2]
NOMS_FAMILLES = {-2: "-2", -1: "-1", 0: "0", 2: "2 et plus"}

def famille_pay(codifications):
    # Codification -> famille : -2, -1, 0 gardés, 2 et plus réunis en 2
    v = np.asarray(codifications).ravel().astype(int)
    return np.where(v >= 2, 2, v)

def score_familles(vrai, predit):
    v, p = famille_pay(vrai), famille_pay(predit)
    return balanced_accuracy_score(v, p) if SCORE == "familles équilibrées" else (v == p).mean()

assert SCORE in ("familles équilibrées", "bonnes prédictions")
scoreur = make_scorer(score_familles)

def bonnes_predictions(vrai, predit):
    # Pourcentage de bonnes prédictions (bonne famille), en tout et dans chaque famille
    v, p = famille_pay(vrai), famille_pay(predit)
    resultat = {"global": (v == p).mean() * 100}
    for f in FAMILLES:
        resultat[f] = (p[v == f] == f).mean() * 100
    return resultat

# Logarithme signé des soldes, des paiements et du plafond, comme dans 01b (un vide reste vide)
MONTANTS_DUS_PAYES = NUMERIQUES

def log_montants(X):
    X = X.copy()
    X[MONTANTS_DUS_PAYES] = np.sign(X[MONTANTS_DUS_PAYES]) * np.log1p(np.abs(X[MONTANTS_DUS_PAYES]))
    return X

def etape_log():
    return FunctionTransformer(log_montants)

def preprocesseur_lineaire():
    # Codifications : une colonne par valeur (un mois absent est une valeur à part) ;
    # nombres (déjà au logarithme) : un vide est remplacé par 0 et signalé par un indicateur, puis mise à l'échelle standard
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
        ("nombres", make_pipeline(SimpleImputer(strategy="constant", fill_value=0, add_indicator=True), StandardScaler()), NUMERIQUES),
    ])

def preprocesseur_arbres():
    # Catégories : une colonne par valeur ; nombres tels quels (déjà au logarithme ; les arbres et CatBoost traitent les vides)
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
    ], remainder="passthrough")

# Plis : les mêmes 5 plis que le 01e, par client ; chaque pli est noté sur le septembre de ses clients,
# et ces clients sont retirés de l'apprentissage du pli pour tous leurs mois
plis_sept = StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(sept, y_emp.to_numpy()[sept])
clients_emp = emp["client"].to_numpy()
cv = []
for _, val in plis_sept:
    clients_val = np.unique(clients_emp[sept[val]])
    apprentissage = np.flatnonzero(~np.isin(clients_emp, clients_val))
    cv.append((apprentissage, sept[val]))
print(f"Plis : apprentissage de {min(len(a) for a, _ in cv)} à {max(len(a) for a, _ in cv)} exemples (tous mois), validation de {min(len(v) for _, v in cv)} à {max(len(v) for _, v in cv)} clients (septembre)")
print(f"Score optimisé : {SCORE} ; surapprentissage max {SURAPPRENTISSAGE_MAX} % relatif ; alerte si écart-type des plis > {VARIANCE_MAX} points")

## 5. Les réglages explorés

Pour chaque modèle : les **échelles** (paramètres ordonnés, explorés par la boucle autour d'une valeur de départ : ici, **le meilleur réglage du 01e**) et les **choix** (paramètres sans ordre, tous essayés à chaque tour). Les autres paramètres gardent leur valeur par défaut, sauf ceux indiqués en commentaire.

- **Régression logistique** : force de la régularisation `C` (petit = forte régularisation) ; poids des classes (`class_weight` : aucun, ou équilibré pour compenser les codifications rares).
- **Arbre de décision** : profondeur maximale, taille minimale des feuilles (grande = règles plus générales) ; poids des classes ; critère de coupe (`gini` ou `entropy`).
- **RandomForest** : nombre d'arbres, profondeur maximale, taille minimale des feuilles ; part des variables essayées à chaque coupe (`max_features`) ; poids des classes.
- **MLP** : taille des couches cachées, régularisation `alpha` ; arrêt anticipé (`early_stopping`) pour éviter d'apprendre par cœur.
- **CatBoost** : réglé comme dans le machine learning de la deuxième itération (même pipeline que les arbres, codifications en une colonne par valeur ; tous les cœurs ; 1 000 itérations). La boucle explore la profondeur, le `learning_rate` et `l2_leaf_reg` ; poids des classes : `Balanced` ou `SqrtBalanced` (plus doux), toujours actifs, CatBoost refusant la valeur « aucun » passée explicitement.

In [ ]:
ESPACES = {
    "Régression logistique": {
        "modele": make_pipeline(etape_log(), preprocesseur_lineaire(), LogisticRegression(max_iter=5000)),
        "n_jobs": -1,
        "echelles": {"logisticregression__C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300]},
        "depart": {"logisticregression__C": 10},   # meilleur réglage du 01e
        "choix": {"logisticregression__class_weight": [None, "balanced"]},
    },
    "Arbre de décision": {
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(), DecisionTreeClassifier(random_state=42)),
        "n_jobs": -1,
        "echelles": {"decisiontreeclassifier__max_depth": [3, 4, 5, 6, 8, 10, 12, 15, 20, None],
                     "decisiontreeclassifier__min_samples_leaf": [1, 2, 5, 10, 20, 50, 100, 200]},
        "depart": {"decisiontreeclassifier__max_depth": 8, "decisiontreeclassifier__min_samples_leaf": 2},   # meilleur réglage du 01e
        "choix": {"decisiontreeclassifier__class_weight": [None, "balanced"],
                  "decisiontreeclassifier__criterion": ["gini", "entropy"]},
    },
    "RandomForest": {
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(), RandomForestClassifier(random_state=42, n_jobs=-1)),
        "n_jobs": -1,
        "echelles": {"randomforestclassifier__n_estimators": [100, 200, 300, 500, 1000],
                     "randomforestclassifier__max_depth": [6, 8, 10, 12, 15, 20, 25, None],
                     "randomforestclassifier__min_samples_leaf": [1, 2, 3, 5, 10, 20, 50]},
        "depart": {"randomforestclassifier__n_estimators": 500, "randomforestclassifier__max_depth": 25, "randomforestclassifier__min_samples_leaf": 5},   # meilleur réglage du 01e
        "choix": {"randomforestclassifier__max_features": ["sqrt", 0.5],
                  "randomforestclassifier__class_weight": [None, "balanced_subsample"]},
    },
    "MLP": {
        "modele": make_pipeline(etape_log(), preprocesseur_lineaire(), MLPClassifier(max_iter=500, early_stopping=True, random_state=42)),
        "n_jobs": -1,
        "echelles": {"mlpclassifier__hidden_layer_sizes": [(32,), (64,), (100,), (128, 64), (256, 128), (256, 128, 64)],
                     "mlpclassifier__alpha": [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]},
        "depart": {"mlpclassifier__hidden_layer_sizes": (128, 64), "mlpclassifier__alpha": 1e-4},   # meilleur réglage du 01e
        "choix": {},
    },
    "CatBoost": {
        # Réglé comme dans le machine learning de la deuxième itération : même pipeline que les arbres (codifications en une colonne par valeur),
        # tous les cœurs, pas de fichiers écrits ; poids des classes toujours actifs (CatBoost refuse auto_class_weights=None passé explicitement)
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(),
                                CatBoostClassifier(loss_function="MultiClass", iterations=1000, random_seed=42, verbose=0,
                                                   thread_count=-1, allow_writing_files=False)),
        "n_jobs": -1,
        "echelles": {"catboostclassifier__depth": [2, 3, 4, 5, 6, 7, 8, 10],
                     "catboostclassifier__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.1, 0.2],
                     "catboostclassifier__l2_leaf_reg": [1, 3, 5, 10, 15, 30, 60]},
        "depart": {"catboostclassifier__depth": 5, "catboostclassifier__learning_rate": 0.03, "catboostclassifier__l2_leaf_reg": 5},   # meilleur réglage du 01e
        "choix": {"catboostclassifier__auto_class_weights": ["Balanced", "SqrtBalanced"]},
    },
}
for nom, e in ESPACES.items():
    for p, v in e["depart"].items():
        assert v in e["echelles"][p], (nom, p)
print("Modèles :", list(ESPACES))

## 6. La boucle d'optimisation

Pour chaque modèle, tour après tour : GridSearch sur les 5 plis par client (notés sur septembre) sur la grille courante, puis choix du meilleur réglage **parmi ceux qui surapprennent de 7 % relatif au plus**. Pour chaque modèle, une ligne : son meilleur réglage admis, son score en validation croisée et la raison de l'arrêt de la boucle.

In [ ]:
def fenetre(echelle, position):
    # La valeur courante et ses deux voisines dans l'échelle
    return echelle[max(0, position - 1): position + 2]

def libelle(params):
    return ", ".join(f"{k.split('__')[1]}={v}" for k, v in params.items())

def optimiser(nom, espace):
    positions = {p: espace["echelles"][p].index(v) for p, v in espace["depart"].items()}
    lignes, raison, debut = [], f"{MAX_TOURS} tours atteints", time.perf_counter()
    for tour in range(1, MAX_TOURS + 1):
        grille = {p: fenetre(espace["echelles"][p], i) for p, i in positions.items()} | espace["choix"]
        recherche = GridSearchCV(espace["modele"], grille, scoring=scoreur, cv=cv, n_jobs=espace["n_jobs"], return_train_score=True)
        recherche.fit(X_emp, y_emp, **espace.get("fit_params", {}))
        r = recherche.cv_results_
        res = pd.DataFrame({
            "tour": tour,
            "params": r["params"],
            "validation": r["mean_test_score"] * 100,
            "écart-type des plis": r["std_test_score"] * 100,
            "train": r["mean_train_score"] * 100,
        })
        res["surapprentissage (% relatif)"] = (res["train"] - res["validation"]) / res["train"] * 100
        res["admis"] = res["surapprentissage (% relatif)"] <= SURAPPRENTISSAGE_MAX
        lignes.append(res)
        admis = res[res["admis"]]
        if admis.empty:
            raison = f"tour {tour} : aucun réglage sous {SURAPPRENTISSAGE_MAX} % de surapprentissage"
            break
        meilleur = admis.loc[admis["validation"].idxmax(), "params"]
        nouvelles = {p: espace["echelles"][p].index(meilleur[p]) for p in positions}
        if nouvelles == positions:
            raison = f"tour {tour} : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle)"
            break
        positions = nouvelles
    historique = pd.concat(lignes, ignore_index=True)
    return historique, raison, round(time.perf_counter() - debut)

resultats = {}
for nom, espace in ESPACES.items():
    historique, raison, duree = optimiser(nom, espace)
    resultats[nom] = {"historique": historique, "raison": raison, "duree": duree}
    admis = historique[historique["admis"]]
    if admis.empty:
        print(f"{nom} : aucun réglage admis ; arrêt : {raison} ({duree} s)")
    else:
        m = admis.loc[admis["validation"].idxmax()]
        print(f"{nom} : {m['validation']:.2f} % ± {m['écart-type des plis']:.2f} ({libelle(m['params'])}) ; arrêt : {raison} ({duree} s)")

## 7. Synthèse : le meilleur réglage de chaque modèle

Pour chaque modèle, le meilleur réglage admis (surapprentissage de 7 % relatif au plus) sur l'ensemble des tours, avec son score en validation croisée, l'écart-type de ses 5 plis, son score en train et son surapprentissage. Alertes : ⚠️ variance si l'écart-type des plis dépasse 2 points ; ⚠️ aucun réglage admis.

In [ ]:
lignes, meilleurs = [], {}
for nom, r in resultats.items():
    h = r["historique"]
    admis = h[h["admis"]]
    if admis.empty:
        ligne = h.loc[h["surapprentissage (% relatif)"].idxmin()]
        alerte = "⚠️ aucun réglage admis (le moins surappris est affiché)"
    else:
        ligne = admis.loc[admis["validation"].idxmax()]
        alerte = f"⚠️ variance ({ligne['écart-type des plis']:.1f} points)" if ligne["écart-type des plis"] > VARIANCE_MAX else "—"
    meilleurs[nom] = ligne["params"]
    lignes.append({"Modèle": nom, "Réglage": libelle(ligne["params"]),
                   f"Validation ({SCORE}, %)": f"{ligne['validation']:.2f} ± {ligne['écart-type des plis']:.2f}",
                   "Train (%)": round(ligne["train"], 2),
                   "Surapprentissage (% relatif)": round(ligne["surapprentissage (% relatif)"], 2),
                   "Alerte": alerte, "Tours": int(h["tour"].max()), "Arrêt": r["raison"], "Durée (s)": r["duree"]})
display(pd.DataFrame(lignes).set_index("Modèle"))

## 8. Les modèles réglés, sur le septembre du train (prédictions hors pli)

Chaque modèle, avec son meilleur réglage, apprend sur chaque pli (tous les mois des autres clients) et prédit le septembre des clients du pli. Chaque client du train de septembre est ainsi prédit une fois, par un modèle qui ne l'a jamais vu, à aucun mois. Les tableaux sont ceux du 01e, sur les mêmes 21 049 clients : ils se comparent directement.

In [ ]:
X_sept, y_sept = X_emp.iloc[sept], y_emp.iloc[sept]
rang = np.full(len(emp), -1)
rang[sept] = np.arange(len(sept))   # position de chaque exemple de septembre dans X_sept

predictions_hors_pli = {}
for nom, params in meilleurs.items():
    p = np.empty(len(sept), dtype=int)
    for apprentissage, validation in cv:
        m = clone(ESPACES[nom]["modele"]).set_params(**params).fit(X_emp.iloc[apprentissage], y_emp.iloc[apprentissage])
        p[rang[validation]] = np.asarray(m.predict(X_emp.iloc[validation])).ravel().astype(int)
    predictions_hors_pli[nom] = p
    print(f"{nom} : fait")
predictions_hors_pli["Règle « même codification qu'en août »"] = X_sept["codif_m1"].to_numpy().astype(int)
predictions_hors_pli["Toujours 0"] = np.zeros(len(X_sept), dtype=int)

lignes = []
for nom, p in predictions_hors_pli.items():
    b = bonnes_predictions(y_sept, p)
    lignes.append({"Modèle": nom, "Bonnes prédictions (%)": round(b["global"], 1),
                   **{f"Famille {NOMS_FAMILLES[f]} (%)": round(b[f], 1) for f in FAMILLES}})
display(pd.DataFrame(lignes).set_index("Modèle"))

**Comment lire** : une ligne par modèle réglé, sur les prédictions hors pli du train ; les deux dernières lignes sont les points de comparaison. Mêmes mesures et mêmes clients que dans le 01e.

In [ ]:
famille_aout, famille_sept = famille_pay(X_sept["codif_m1"]), famille_pay(y_sept)
change = famille_aout != famille_sept
TYPES_CHANGEMENT = {
    "Sortie du retard (2 et plus en août, sain en septembre)": (famille_aout == 2) & (famille_sept != 2),
    "Retard maintenu (2 et plus en août et en septembre ; pas un changement)": (famille_aout == 2) & (famille_sept == 2),
    "Nouveau retard (sain en août, 2 et plus en septembre)": (famille_aout != 2) & (famille_sept == 2),
    "Changement entre codifications saines (par exemple 0 → -1)": change & (famille_aout != 2) & (famille_sept != 2),
    "Tous les clients qui changent": change,
}
lignes = []
for type_changement, masque in TYPES_CHANGEMENT.items():
    resultat = {"Type de changement": type_changement, "Clients": int(masque.sum())}
    for nom, p in predictions_hors_pli.items():
        if nom == "Toujours 0":
            continue
        resultat[f"{nom} : retrouvés (%)"] = round((famille_pay(p)[masque] == famille_sept[masque]).mean() * 100, 1)
    lignes.append(resultat)
display(pd.DataFrame(lignes).set_index("Type de changement"))

**Comment lire** : comme dans le 01e, une ligne par type de changement entre août et septembre, avec la part des clients rangés dans leur vraie famille de septembre ; la règle naïve est à 0 % sur les changements et à 100 % sur les retards maintenus.